# W3: Does the loop know where it is?

Trains three small image denoisers on CIFAR-10 (M-on, M-drop, M-off), checks that each learned
to denoise, then records their insides on the registered inputs. Pre-registration:
`docs/W3_LOOP_PROTOCOL.md` in the agi-semantic-core repo.

**Run:** Runtime → Change runtime type → **T4 GPU** → **Run all**. Allow the Drive mount.

1. **First Run all = the smoke** (`SMOKE = True`, about 15 minutes). It writes to
   `MyDrive/semcore/w3/smoke_<stamp>/` and stops. Leave the runtime connected.
2. Claude checks the smoke from Drive. On an OK: set `SMOKE = False` in the next cell and
   **Run all again on the same runtime**. That is the flight, about 4–5 hours; keep the tab open.
3. If the runtime drops: copy the stamp printed under "Run", paste it into `RESUME_STAMP`, Run all
   again. Training resumes from the last checkpoint saved to Drive; finished parts are skipped.

Nothing here downloads anything but CIFAR-10 from its home at cs.toronto.edu (md5 checked).


In [ ]:
#@title Settings
SMOKE = True  #@param {type:"boolean"}
RESUME_STAMP = ""  #@param {type:"string"}


In [ ]:
#@title Setup: GPU check, Drive mount
import json, subprocess, sys, time
from pathlib import Path
import torch
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU.'
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found: mounted the right Google account?'
W3 = SEM / 'w3'
W3.mkdir(exist_ok=True)
print('torch', torch.__version__, '| results ->', W3)


In [ ]:
#@title Code payload (W3 builder files, verbatim; collapsed on purpose)
import base64, hashlib, io, sys, tarfile
from pathlib import Path
PAYLOAD = """
H4sIAAAAAAAC/+19/3bbRrLm/M2nwHBO1kBCwiRlKR46yl5ZlmNvLFtHUia+y+WBQBIkcUUCCABKlHV0X2ffY59s66tqAA0QlOWJkzN7VzyJBTT6Z3V1VXV1
VfW1H0zC6+Sp4/iBnzqOHd385Wv/OvTbe/aM/9Kv+nfzudvd3dn7i9H5y5/wWyWpG1Pzf/n/83et5v96x0kib/xHTP9n5r/X3d3bq87/zt7j/P8pv2az+euO
0XwVeomRzj1jEYaRcRmE18b13Is9w08NP/nvzT5/TOZu7E0MIIphTsJx8vTXHefdhw8nzsnph/MPhx/e2cuJZTcaR1defJPO/WDG5Q7DhTsyEn/iGSYh2jKc
eAvCtJZBL+Mw9vgZGYMw9UZheGkZbjBR/Rm7Cy7aMN3AXdx88pzrHS7gzrwgTeTNMpY0kZQUe54RBsbCv6IBYQR9I/G8SdKiOj559GfkxvyS4kXa9BOvkYzn
3mS18CTND6JVahBmpH6S+mPKmfWHIEA1p08SY+ovPGMaxks3tY1gtYxuqOEF9SsJAbV4FSRUjzEK03kjWrhjL7GNc22MhrccUceoVl9VRkAbUYtLAuD5myPj
zcHpP47Ozo3XH06PD85tg4FquHHs3tCYMC80IX4U0YyMbowLPJvWheEm6M3Iw0zZQfTJiNw4TYxwaripsQyTtHH25u2Jc3hwQsUICoaJcX04PeTa2gCTsXCv
ZcZfxQRIGkbqzWLqGkH20l8sEiP2kigMEo/r/c/v7V3j+CU9NkZu4u09swgI0YKA4C5ChQJTP8b8rH2CZbRYUbmAmvHpn8iLuYsA1sXSDfwpwdf+jyQMLmzj
YhG6E4yK5iMx3AbGPXLHl0YazjyqN+aJobkbXyaGxwBCXU8AGre3u2cbZ1QicJc89TcGo56RujPDPP7w6ujdmdVvNAzjNgxakziMWmHQuXOk74bx/d5zgwBE
PVsSrhGg/s//3rF7lm2ceHGb0wAZmkFCmqhFw5+2KOvsBQ8JGEY11/+mNKx0p2eYQct41jIm/tLC8N++P/nl3Plw+uro9IVxEa7Si2rO7jPL2CckDCZb6y6w
FnPDGO6l12F8SUChKoHYXpS0525KbaCmciM91cjW+lGkphFqgpfSTQattUNwoX+e2BqACbhzd5lVxTSF5oYhrQN5hzAoJrYEoHAmnhGafIwhnnhx37ggSF8w
qLdDgqagNDbAOQMs3mmg3LnjOydKY9enJH4GyeF2aW2DLqBl1bNnVj+f9NJcS0PdvWpD9+OAyk8dqZ0MdNB4Hca0WqhSIKhgGaGpYUqXt9YPYmRx/o5tHLdR
mtYjEScs84RrU6N2gtViYTxVSTR4STC9Nv5ubYDmhulahgFE473FC5lQmUwiAynhGq9RJrMgTj4Bj7MuFGB5DsKAll3suZNSE4D4ngL4CLWYQp0n3trotHfQ
8tnJ6dHBK+fw6P356dEZDTibmIulX0Bz6yAYyr3nGfiddXrRyh47G5PxAgRpduXGpS/geAcEMCaTIEkg0TTqmOnDTo8+/4+zD++NkccsEENgmtw3Zm7qOQXx
mU7vmPAB1br27gvj0rsxmpGbJE0MjOdqM3fDpGyJ0fSicDxPmi2juQgTIs1Nxk1OpbTEGzv07mBGUNts4gmVbRlecKWexkQesULVK1H51dJz0owg243jjH5S
56mfgjP0hzrD42VKgH7S3DKtTQRZOUeHP6+CLENKwifTj+M21QWaRg9UkYmtiO8uCGEmlJ2WOxc8Pz14+565eYtI2hVSpSKCP4kxDX8ZhcRG5m4yX/ij7NUP
syeG1TQOlwT9FFkM9eGEXhtZLuHkYKJRo/E3o00/bj32ZkTwhAk+yVhswt//uV/j7Ojo1RlR89smj4xkrF6nt9ft9Do0XQenx0VClxJIwnp5VCT1kOf9wbt/
P3t7VqTutApEb569OdDqIObRlLVSpO1S2k8H51q1e8h1cHzyTkv7Xq/01emHk+LT87vG2fGHn48cjMX58Pr12dE5jajb6ZAIIw+dxq9vX52/oZe9Z43jX94h
2ewyn+lZjaPjl86rt8eU9sz41uCcjaOTD4dvAJi9jnp2jg8+orZep7J6/8ZixHQVsyiQEvcngcxwMzZPPM6Yuv5CiA4Wm1pZVuPlwfnhG67zeePdKT30vPaz
xq8E9l9O6G2X+n347i0eu3angVE7eOnYvUZDxokemgAhDbJl7HQ6qPTsyDkmJKXqOpu05m8GxCOmiUZKiO1DesQyGmQFW8a50cZSIbpoNd47J0enzuG7gzO0
RdIMUhhXWgY9HR68A9w6qvkOfcVkcuP0hmfn5cF7xjHTpEy93Q4tfOozPdMA8byLwt9L+vdIx4xZVuPs/OhEK9z7wtIlmswTrjrJRakMoKVyvTl495pRJUt4
z7hSBz5IHSQigKyBG2QFfqYCz426AuMwSP1gxYtWynEVWUGM8oxnm2AnaH9W3/jfwNYiktFnsT/higTFTCZB2OfEHu1kJlaj8e5Dy3jzVnClRf88r9ZElGvi
j1PeiTR+Pjg5OSBIn3KB3c12wVRid8yyt2DwKmIAhEQZQTwv3ShyrQYJs+cHzj8OTt/yGKjxDdyD3IYdXCKbpYk/naKiG2wMFp5LSbIXoSUk9TRAQvLO9Xbl
PUNIms7Gyw8fsnVOGPfq6Nw5/wCs7Hrt3cbBu5M3B8gZRDaxzSQiYcFs7xAOEKZ8b3GvCJwz7AFoa5f6EP9oSzghPjkLHJbqbanEahz+w3n94R3j426jcfb2
XBCr6QXjnR6YHj1099TDc/wl/o8/E09e6a98pwcqYXEdID5Cg6WevtCgvDqidBldymouJ6ERPaUsaEjj5RLSj420ovG7xsnp2+OD03930EPqHDdCvIj3UeEq
lu2pbOmUaMzyFu2nYz9NPZIfgPi0O2Za0xJxi5+thrbHYPilKWASRfwvP6ckHyicMdNWisWdPmmlT7IH/ov3hmyhuJowYNCSNIC/RMH4T9BRbx2Ct7AKBraQ
dQCBPmtkjlJ2VFLGE5+BXjSF3OGzelVsC191lqcoCH15nvM752dk5FJqjdN7F5wO2ItnQt67RuPw7euDU+eXU+Bvc56mUdJ/+vT6+toeJ3YaxgTR0PYmq6f/
eRn7n56O/akbt7ud9sgP3PjGTt3Ynn1qqlqOX+2iFppVtzt55o52J52dKfGaZ6O9758/3/GeT55/v7vTaRZ7cVpPzvNOx8FCajQm3lT0FmayDC+9/dckuXu0
V8UoYy9dxYFxe9k3rozvDHOTA/tUGMUMkr88o2Pxkr5sUXaS2VnssGnHskxM6y5rC8qRmraYYZVmTB42pk17K89fwbKKSZSH8lop5lQeWqXZzB7Lk5o9VuY2
f660oKYbf+74Swanfp4P47VX0YQIrcDV0kGOr42SVCi7mkx5ZJiZCsdgOTNTrkbhtRc/rVI2aK3q5ELMh7uI5q5DHMK8pDr2m+MwIUm3qWaF+o1kY59QDDoU
N24WQ1B9JZo7JlE2DicmSTAkVYAIk7zNRJjI8zPwpg4ttnNLxpgwke90nvPbVMg2tWuaJj24sRvMPPOcEK5rkTh/btFTgiezK0/fIn/kU0qPXr41ZM818lI3
UXUt/Igyt43poNsfUr7poN/uDqkf6Mrf//73Eqw3+s81ERVpHLyk+goA5bAhzvvT2fvTnN+YlJG71zYOXlpWMXFV7d5XmTY1cbxznHoujYEW1FpNGG1PCp1R
WXcSUtsubR7Bnef+bN6ext5vK+I0N7Qv8+LZDRFzqjTwFm0XWWfY72C7g3pnNNi1vfTcwIRqbb8rIJysBQjuKMHcgdGbsxZr3/Z7lqUVYElcIcDkZnup7tZS
CzcqikGInw36xN5pZuXfIYGfk/rtXiWlpzION3fnbb0aKjksp/T6w239iUqDmE5T/J/M/WmqvffMmYWReXlZDWVpt3xNlcxsEpYiD7jKyTc3LWOtALuEGDjo
U87+tXyN5cP8JgpT8+aGujs3nj4Ff6MybaoQL9JDbwI9135pQdISiO2luzaxrGh1EndSy2GEPfK+MVA4HQEIZmz8uG8sQsv4b3j+gRDHGpbQQGp5bpUgCy6w
CFuUG3zgE61G7opahfJMo7VkREBibnltJ+lEAzQJ8hZBTwF5vTER6vtkTf/fQPggCeRbDKMiHCFpIOjAj53hsEIAaKWML03uSI6HxTJm5TdE7/ZX/Qk/ZO2x
wwpyc1RmvErBYEse+mrPvfXEJ+ilpqX4qRNEn1Rp0QZl/HQFwuqH9kt8e/vBlBnCWGl1fzLpOwHrW1WmxHtWU3vmpVfuYuXlzbDOXzK3WMvdgop34sctY+xG
+5lwUVChX2OoKi+kyIVhgvJgT+BvqOkt1oFQlTZ0Ve/f81ECjin0owTjgpoRFScPVs5JaEPiyXGJH7OOPzHSUHXsaUnDbxunPLpEaVvlk5TJiZwUJLBBT2PK
m6V9sZeX9K9J3cR50P55vCIweDSE1Akv+VUBGYLMwgvMKxGIWBoSSNgMVpaHkNNNEi/GnigwAwtcltB62rwFNO76mXpv4if5cZMMtQCecRvcNfNGAzsKIzXT
cXgteEHpyWppXg363aEdSNK2XlkGfegKefMACVAK6hPtDU2aACK6xD9pQ0U0Jq9fUWdAsgXZD8ecKZazYrj8+XqOcyf59AP1lZoxAwy4wxN5yY9WIVskaQj6
uvQDKESl3HfoU0FoRvRdw/7bS++G5NQB5+2j/FCkUe+mVRprJpAWNUnnMAkjy/gRCG3I0QP1oa0a/5EYgr7JBZLOPXfixUmfEJvw+hIKR3+BI7mpnyaN8lEJ
Dycbh4KqqbcAuebvAGy3UyamX2+cU+AV1ZZhGC84ku47vckdVl0zz6mQH6ITclo29nteTqXyfLJ+ZBmat83ES0nsFfLQROX0RhjRxGqmx6n6ggMXepUhtBg4
w9YWxXmT26TcMj3QLzM1bParpHNrDQIPKiCgo2Zj88qepDeRR/UtaPnSa86Crc+AU4OnhuyX4IMylPwzCc+C4v3yfMaee8kpRIaoaA7qZoliKZK0IOGbFco2
DigTk/LYOL1wUm+dYsFSK0hjMkT9k53YYKiVHqx5RGsMBilUYj3guRoaf93n6RpS93kus36pCedGuPUJiciJScVbfCoSpPsYbhinDs4EhPiV2IhUJ9yDz1Zl
8qkwEdWcTZx6IIHLEa0/Nz9kpq615KgV3MJzx/PyWavhzlzi42mJludUnFvIiLg0V1AoLELqtDcxTQ0qGnylxOZsWBtQz2G4LzCsR0CCzv7CXY4mrrHuowwv
i6GVba+CUHEhbV/lYqv3s3dzFMdhbE7RjMEr9q/xHQ7jUYZ6fZt19Umpq0/umkpA5eP4/QwZ8tFWmgN9yUdNHeTFOlTjlcVl6ThdWXZAIYBCVuWwjOsylH+A
u2SDuV0PnqCJJ8O7bAkrBWEimyJ9UpsakfbTuey3CJc0uWZksQDxqdwwjz0nTJd949PgUi1sFkltFjDurKqeg3eiwZh25QF25oMoKxbxKSRqHVpFPZxA8mSm
3SCJrQbPZZ1r+Lhlsat+lJYziznBjVnFtlrkrRIHq/Evaf+VmeX8yfZf3b3dXm/T/q/zaP/159l/KQuR7UZdBo7NYPoCC6bxAtyByL4frJLQJ95zCiHzIE35
FNf45T1RRpwGL2jX7U0a/zg6fXlw/vZ4i3ZltKKdZ5prV+SMHyfRRuovie+IfG2cP3tBzIjtcUQ+VwZr1IZlN37dkVYnsKi6Dg02PkvkCBD6kxl0dVTrMlql
YshkViqjlXvtxhPD+23lLvz0hjalKS9n+ssUw3QtbhGHMVfqSMkVbpd4V14gRmXYHcFsgfgws0hsjttRGMLaAboegzf8L1AfUWk3bxan/NLd1L3MjPFYb892
YhPmuUSSY2+cLm4MMFpYSyjbCzlTbBmrhK3BFNEVqw9i0AvPjXHiHoRtqfOKaqGhiYUHT9roRmsSRjqlQ/WlfkROJcfz0osdBLxdDKqp9nQV8NmVu0CG1/nm
/TMatypOfNnmXeGlmYoljpD7ubvA3psSWBeTazmlq946MtsYJWtZ+DwLor98VMpP1EA1elf+2NtPbXkA18AH4VkuKrTZNsS0oKp5HwYkxn1rTAf8pDMUqZuY
mjlwbeqzCZUUa1ytIXdctB282rDCTALncQg9sxoRKxqU3baZeOic2zJGLSMtRs3bnBXt0EzLzvMW7Bul7AAyIx7GXexWA/unOFxF78N4aT6nKqHrCezDMLjq
TUxpYKcFsbpbqSeV0u9YI22iE5SZhTI8iwgMKFRa72Wt92paH5VaH93TenLpR1JBqatd7oALeWgkXaAcbyEq0xqHIiWDpFqICpBrgiItPA2KcwipAiXzNc3X
YmUq6JlryyrLYgINJRNiyGUxCFXNoTXnfCY3xCpSwReFNTUt98ot98y53nKmn8pqBkSobzkOgUA/BInGX4I7avJ+u7xST2HNLI5LszimKaQlMcbkVNI57Z45
cfYBGq1/Iy6jVLZr2S/mH39ryanXft5HBTjMGIlknNsUjEIt1Klry14FIyJHpoZfbk4mknCaQk3wm50STUiiMPFEgWz8G4meT5lO2slvcWpiyFjD7e7GBK2z
CQpN84oKutXKSn1Tw7OKeQSne9A8+gG2g4f7e89axnK1SPdzwxv6uAjHSjv2oNnmJSxmOofAcS6vDhjLKHGYrWfOsW8cqsYqxIKQXjDljM87YPBlFtTjUFEx
xo8z/90vpjyWiAvnsCq92aQwflAmDGMYeh7W0JHxPFF9J24EczisvMHhkFsWaL+DXoK6ctjQteo+wRdCvxesll6MLQrAbfU31O8Ocgk36VU+MzYvLqWnWmMD
kH6Y8B0S6LNBD62NstxZzrPxiQDkY4cCZQ13zGgb3X6tYoZ6kO3PmFyM59ZmUzmIsqxUqq5D+VaPamlsdOiHz/Vno5kKYLQZnWOyeEZxyETgeXB3uJGlP9kO
d/yvcDEDCUu85Y/DSpWraKNGqxZnYpgykthmssqrgkHW/Ti082U4RHQHUGBt9L8cPq2irdjE7f1Y1djVlNxEkF8iMdwyk7G78JypC7l3v1fmOwX2dKvYI3R6
ldaRqjKLm2/QKr16kJys/nuFDm1GFYnMyaWpC7ZCaoX6aeR2kwRS6/Mkq4ZIIfG+VqZ84u/5spiXERSIRJiWr8L+xpwk2IW4wdjDlA06Q23UNYgJfJJ8ZoUc
oNOb+fm40r3xYnajWVzWIxhq5VzmXIlsZVFr2+jymjMCsCmibam2CptV9EDIZLh4D2y6QzMH0UNgNM8lE95H8FzL+i62EF8drkqMyZYGIJvv67wg35PmWzxs
0X/vIW0u/MhG38Q/hb6a9/7fKT8MZXxB7+hFsXuG1bfNBuss9oiuYGOLLFtiVlvfJ1ftfF6uEo+NkjlVrYyF6kQk2T9U9eGfrCr+V5NF5WxSVjvNJHYYaKci
ltMYnFzCOqEN7NJLqVnBlU9eHIqxhiIgbMW/bX/mLIhDLUBCcnrf0OxiAXPoT8de0lcqhCQlvEL/Rh4hmyeKEJAQZUT71FippVA41hWlys07XjB2lLueJEw8
laBOMzVapviqWp0FL81pmPV7iVilTxlx8cEBrcZncgF+RVfKRe7t+iqyfj+NqUDv3q5v5Mq7Tl2pKYC8DrvZgSCtwHCnMJe93ekZP/5oXN41C+28iC45Llkw
VjHFWheP/6vU8awu6pCqyyxJHfT/pXVv7cUyZlKg1rDOZ2mtn9dp28zUEo2cOBbGcN9ZuQuF27E39uBlmp9wVUnjVpZd9IibzHq0L3t/KqWeeBEv3eSyuulV
B1QF268c70gnSktZyRL415/Kn0I7IUufqxMYpSV1Rt6P7RoNVb2QF/YcNvNSuRqsVaZNdhqy6kOd+UIFB+M5NxGFyBaug8T75KfPQLEldk0bVFnhQBj7Mx/K
SnAUW9UubmyMYmJPzYdehbaUVZgylbnjcqPkhKirjMVRgDgNgSlQvWF+ZMbKFgbH/C3jFiX7hvmSZcs7yyohmi4bStMpD1obrw66MWxf7n6/TPjHEtd/HQlR
9V9s4DClfi7uFWR9szkC9KBCDwdVXsDei6ZvDYfQAorxHCQ1UwznvpKQmnW+ShnGA6a0D2j7j2VG/08JvPdhQ84pvwAbcpHqO6Ob6QI1lvtQDJFtqS6GV2il
EBLqhlV0n0k9pauj8aV76cnJr5m6s21CLVwRdCZElAg+9LB2KHkm3xmmG1Q8PME0tFZs27aq7pvGr54/m6fsMquMRaoOoBfowoWcDRaOoAVN9MVfop5DCcIs
3YC4t4NsJhfWD2PUtiIX7eskcUAI5wjs3mIp8Z4TYYjPvi//Umf7X3j+ryJx/NnxX3Z3d/aebcZ/+f7x/P/PO//nAC1t9k4HFtxvB9DC+Tb+QSyEiZu6EjNl
gg3mklYuXBuMESInkLgUe+6y1ci9JFly4lPdKGSvflBTcTNvFc7jyAlfx1bF5Y3DBbT0OCz80hAuE+d7SazutgonUHIBbRk/tScwY8tyStNMQ9vs5l4K0dIw
xwBMDohf3tp+dBOMrI3YLRzKJjOg4dqLgEqFIoRjlbgcP2UVmJatx8nhaDscn4Ejx8Ah3jg8+aUFM1iCJLUzD6/LIEpWkC3RbTg43+cEz67v6jlMsqdkvkr9
RX5k78YYSf7qL70v85a/zzpg0w6Aq7azShW08O1MfcoAauQmCBkPaTTOD37a5nao6aKAm1/ZOSC3MVhOdh2AywR0MgsDDs0izgH0Xen7GeVDEjA5K+xsR022
jZv2S/LWeL4KWNzyoagR48S+MWXLMbNr/PCD0YOj9ajZrIhT88xhjWsocbV5jWsCbNIc9l7U+0640/5RuTO3GKekjyrKhbJ5p94N2LEHezkcUCbsBM7O3S12
3dyBuCD2Ey0DvD7n0Gg7cuPEM2P32upXNifwzaFpH61gcogc7GCyIhrxvDgN5aZJYNrpfN/bEHYgbbGD0LCUv+hXMRsK1e3SrPRnn2ReUm1iUpisDqTbKe2O
1yl8sXnip83CBZTDxCRwBX0KpHM4wbn172xKUkaqsE9lubrQUlD39qyhtqejxmrbqm8KsyRNlZvhClckQrFEy0auhEmu7SbYaLMblEwrbFe6PURvaiPMARvQ
h79R3qPvd7o6HqEys2KEmcbw8uEvqZchF0cAMtcdDsPiRUmBXWuHhVUcjbsjh7Br3YHoi3cocSQNwWJMpikt9nRgh7kChXAOf2YfvByk5TlW/1k1I9RHgXTp
AEx7uP0shbuAVPS5rpqS16fO3AzTZ2MSxAzi4BMLBJ6AX4jEP3khPMZlW+oURqMSkqXiRMjQ4w+OmmJTIp2o+C2Bo5Ymf90/szmAhQLvLN8U/QSfQZeeibvo
8i48RVjmpSFicdLQkSQ9sTLPkmVeD+HnBAlm3u4sq3l/ZuWGyxouq4ywreIu6uoq9kdEdQOf2ufPfdMXV1L1rulwF2zHU/TD5AzlHhg/IFpCsXJKBTAywsAz
+5xIkRS2agbAm8woKRUNssZysrGl4I3vwWYfTm7ocIbvahlwJCXoYCqzyOGQ/sA5LFmWMez0FjdAeGZLYBPVa5KinHG6NpVhW750p1F3z3BXaTiGjp4kk8Nf
Xh3QTi8RA03rBYf/wcZSYgaaJLeIWCKy2tJL3bZUKrJfsaLHqwkWtXy0sep4L4Xk5uaIsj6oHjrIvy+Z2QgElfH+tjmOViQQsGJxX8qqcFuIdoS1ONlH7pwn
xo6bMqzKrn5n9rtTnID7AZyQ2TXIizIX6DNbwsXoxCETzf4AiYN6GC79sQNHQTMc/UfL0Ng3HjNjdk7m1HQZMUdJ5zb4npMQd/XXnMGWZxpL06ZsyqJfGTnl
LdAX5WmXEMHlaF8mpammM5rvXTscVkU0CDI32EMTgCS+BasPzmyOPDBQ0Y6GmRU+y3j7NYqIbAt+ZiNskKpFdv1pmCGp9C4qCAA9+0v7YOIufzWVQJ6dfiU4
l1/E+5jVlviT75sd+++Z+znJ2qyHgCrGvdnv2Mrl67NIyrYFcd4FWkf2T7E7OeNk8x4ELeOi7vTQ5L43+wIfyLYR3LboX/haccXwteKHlor9RQmdIvpXnw/H
ynG/kJaHfMCUOYhEQ1hdCS2Rt56kA/U8tLUCVt4hZMBT5XM58ELRX8qtXjbqy4aAPPI81AeDZPUyrFZfHiI3oiflQ2bRV1AVW0eR/ErUbpJPIju0SI6lGzmI
Q4r9y36G3oIqiYMwe+o0QXBBg1jRngxzkn8q8grwanLyhyJfDraarNm3IncOwBLYakADvNayT/Tck2pmHUeSTIMoYekk+ByDlZfvx5b4Xbay8KsZ3Gjnn53R
TBBc1NlI4NO2/d0av61FONuPYmJ6uTykCtYxVJzosERCg37CYH8ylKMaJHB/KSENjQvu6IXwaN55f4RzQLb1IXE34QiMxoWM4YIDFhoG+1InJJRKjNpCq6HO
C7k+8RRScVTHIW2W5eOFNta8ac6b5sdHXjDhaLIuyxNwwYvR1WWmttA8s/j0vGgk56+KejDZyKmUhqKtAgdbJTQT52Ul0+3nT9Ck057pY4ZpIj9VZT8R1osD
QQGhCdG9hilDxlJG9TpRF+LNFWc7aeWtnOMriTA8d4Wo6anhqQx5OstiwPWyYJYMJKAdwh+UpTMxW9dVuzX2ESn2OSHzHpoVG/+AlOlGFnivGZx2yGJuCJFW
+dBlc1eQ5EyU+13eG1RNBmcss+wTaL5VE0Zy44ZRYcTafEIEYZfODLZtNWca0YwoRMx0Q4SmmYSV0vPQzmrf+DigcWmMeriZRT8txvi13IOSKXx+frzu2Mho
Ij7suuKQLbBj/4uiHk6rigu6Wbc7os1kbWvl/mKSXdkr5ptHtWssUrEB0EsFSxZsCNsG/lDrhXgHAyM3DuFL+JUrLGol81KwVz4KEvFpnTIiFefewbI8blB4
yv3aXiaegxecE2VOK4KBpfzACmAzIYWLo5PUSUMnoI5qwRXy426mHzb/MVG3ZSNeMx/S1+ZcBWKZaVIrlQnNdIfQUiYcPoh7QE3HS6dWuDuzEbSxvkeEpptt
ZJ0QBVq1fZEqYVmGkdgTkhjHc9PSwg2E2Na2eYUlN8G4b7DOOQzGXhF8VT+TrZUh+zXDxgcbVc7jMPA/6X3TWXp2dC2TR33FxoRXeTl/mY9npUyNbBEqp50t
pTNyCyLynQpEofixOW0axi0RyjvF7245y93TW6bMd33BtlvwXOn0kyGCvvTtZ9O7lqTrncu+7kzvjOSpBKvVAQhpoQyv0r6oLNSKtLFxjlvIHcxxTX2I3+hC
iETCYLMPDQj7wnTqLKxYoW6DA7PaDk2ArcRm0aRV2XKVFlqxy9pWSJeadD7pXbkL09oQ0DSlVelkJYuD+rvNMv9NrVJFHCylx0qcOeGjkj62muWI3JwzSCXL
Jfu93b3cq4W1Btnmiw0e4US9ufgt5Y+XBYapuNhrmipi2pBhYPIySrRZXBcCC+Qqh/WQ5nrg96GqGiVDq46RpLWlEBwpYaW5mWoVWDYC4ptWhRekn+EDQaGY
o+SErdvWsJDSZalRGC6qskZmvVVTp/pSoT3FYER8qW0JY9gq1QD0GX1R/AgeiaNi7jUDtGK8lFDAqcTAS3ws41I27Z/pXwG2tRE4TlNToz+ZpmLGfhWClB9x
vHiVb08CHSOLHcSx59JK+m3F9154iJWAjQDkjNGNRFQNp5ncqIe/GMGzeOqvEQZ7tUh9WL1m0fihIldWDhHsM0RF/y2sDaEkZ4av0jKNuWjIOdoeidVXbDfC
KbmgHwczOUWB1i9c0mKYutSwQ+mZyMjhFbMQF6ySpY/2eB4SnHmiCSQWQ0IRIX1bm6rsKsZ5kitYAxXUjEUv5IAZ0ITYPbNpIkpmUD4cEkyqquhZwJLTA+oG
ZMdhLg7Xk7kIU7K/QWsg/LB8xfkrThFqkvGPVT1RUNHssNTMyrEAwE7iXQ53q+6IoPZEQrpOYhZXa3Kf2yLjI95cfey0vNCCC6FP3wJADysIrKwQvyLo25ld
hIXu6ya8aCnVosqlElVOC4xJtWohhYDyUCdJ1dDUBE0O1A8bMRuRrXA61OShK12SLF1OgRmUClhnbY8QhKxYSE62hEqVLB5WCUfP7xugjptt0yA3q6qEH2ny
0FEF/rbyGt3FwhwN5E2CkYzYvgy5MPSrvLdXlgaeoKocY/rGJpxZNxk978rcu7B/4HsoLOMrxrpDBGGHqjYzWhiNsalE0IJ98YTvP5TGIGz9sAi3RjUR6oK2
nNkqarqiP8tZmf5Ic9uoD84XpGUv8txUApKCuRbVcrcLWlWNGFOiXVAUZQHXiYgh5LoU52kMZaGoiodfh77l2EQjByYsZxoJ4VsiLGXA0Mdo6z5C3Zpufihj
UwoNqIktC9Vi1VUjC5f+zRXBc9rvT/w4MQteCHPDM1tF53/49POVAwpiq3vgxX67paPZFRFZCYjpLmY2spqrLOZjy7j0vIjKZAGt6g9o2cAgDkcecLkYjJPG
+Jc2ejlS734ZUsvVC8PPYS41hBNlaemLWKhW8vPYVmrm84g3Ft9FKlWCt4aIgz7lGW7FuO2fc0ySLHdlVNRboM/9e1rY+rlogbPc5XEv2barRLN4Bwnii9iP
48Vq4v2zNEzdkqHmewwhj0tQa4iI1Z1o8ZClBStvUyN9eYdyCli5qqYencY8hHoyOMr7otPB3OFlowVLh0uJNmbUs1JggIyIOlfC03aeD3c1AGn1qxs4wl7w
Z1JJdFLx4y+gkxXKp3irsiP8+mZi9TvjPDL0Q7fGpd0wCftnJRdHU4su1DKmEUx6MtPG7rO6q7rU1V+aVZbaYD9kW70hg2cxgm8TnCyKh4xwT76g4e6/7Gb8
81vpf3rP/rU35N5I9feL9t36fr6lbPQ2t/PeqLw/b2n+TOWDDh0tKiENUGKQ5EpJ9poY3rfRr6oZzuxy1HVNm10pXqYyyUZgQRUZOFExRrlreYjRVq1qYbss
okiLXCeYqx06uBtRO6Kkldoz1vR/bu/W/8y1GwJJ7R6PfDHLwUtnkEQDpqAqoDb2xOso31+vuQuDppz5RQOmi0PwivLnqPxd7Y9HGSLYRVAXdR8J7olIQX35
CEI1YcntIkiMiootuW1ET5VEkV/z8sgqVGTG2ngEg9ygoSCivOdH7/IFm996YUoGhpv0tLj9gnUAcaxqTqJSOEqzmd2HUlxpArCqctvRmmokFFZSA8cwH1D3
qe5BZ4gPeQulaRzmMc632TGi3iZhXbO27u7vrBiLqFJzdWWtUz5kqSRG1udbUGuOGqLVUcsa8+VC25HyamkZq6o6+CHcautq+NxKEFHGyU/UMqfYKl/J0HaT
j+R6SlVV9KCqogfUpdz/VxLpCocR4iFN3ZUkC6HK+ZJNiRXIPqe3Kv+dcZUUHs6UXi2sopYrZ2EH+0gT3W/jxZJ9GXvZ5XsyofV5qDxtTKsNFj1tqYPKjSWc
Kn6yHAnFtvSVSXR6ygxi0365XuIorRP4uuWJjAfQqmY4sXHfFl87Og0XC+VtsXH16AuD7yFOiI2qe2f9dBPFdYxWt4jqSA198Ya+eV2oQDsD2vMorEVuhaN4
0HB0OzzzMlVVqFzFmVZOBTb1ogJ5lqXz1pVUnvfswWSwbva6e1btXN2bLydRGzToYYQnuzkC/kJf30xzm9APt1hcHbnQpkedGcy8gGX83W6vkAlevTo5LgoV
vS1uU554Y9oAJ+wExGfPcmWMZ1xwvRfwJec4oxMjCikp9sO4pYhim0OiLiSmdxpmDh3WC8QthjkTD0Eze2ITXrafSjIC7MrNYoixvLmTWOtE+PdaAQl+/xN7
B22rANQPvEr4rUqwCDn15sgWZfEUYUcQmJJV2zzmSwgq2it2wJX4WS7xw55YlyTgjfSnN9wm2E9p1Zm5YA9v2y+Q7h9uIuJVJPithyRlhlOy2DGpLMFo0/AF
ZySZeQwJ3QvqjZyDVC5N8FKcuHBxlCjfy4vNLBpxe0XNXOJp1mJhesMJ+Kj1QjL1rEr+UamRKxeUjautK1HK+6niG7AeCZssGdJjBd8zOYyMPLLv0HbR209a
EJ3chG8ZXuEJJzhhoNY2Y1m7O3zBt5iqBexLhGDeMBTnOCU8betY+bXw7fO49jvwjMm/Mk3RMQ2raAPbkFiLcVZ9kF2qPFcfsr+og0ucMpIctNTNwqUx0eTW
uGWUQVPvo2HlHF1HoGCbY8kWJMqYVw0D4WiBQr/kCIPG3ub/LXF22dyGsntYRbK+lHH/+fI1yS7OFkkwk6tFFJ3XH/KWMuXHuCr1y9zD1h20YfIJ66ZD2NyS
ownlOfZV8EIuEZetlqbXSeVarVLnKjxP5SvdyaOMcCN7FfgkB2B8MW6HNfUbxrgHAp2BDw7W0RS5chEu/FpyIFF+yxr0S9ezrZMt6jWljl5n6jX4h16qPWHH
qlXVTdmueBOzE1022mLvoRugERxpPMWlf1SvTWmmajs71N080m3yfl8eh0ptnCOrEnVz1M5vep2q+Ck1uFQ5iVPXyeOays1d9EOLd+qKd+Y1xWurhP8EIISL
evhB03yrzcsf5B+N6+VrdV4bVGPQ7/ae30s7JIdGQSRBrY/uZwlJKXvvi7JP1GVbgobqyrtpF9uaNtXFfImv7dvcilb9edy1Q4UdHCHRdEyAceGCpze7N7mw
LZgYP+xrH7KTWjH7dxCtAO515sfCfImNJxOJXfK82EpwBAGXuPYknbefq2iHfb4hIXN3mGaFYajowqmlMGViNbG4LhbuDS01MDiiQNyAg8IU9FZcilpcaslx
/1BUQioodxm5eVruTdPuVyj8AaNVTmKEYpjsKqWfq0lnvzWUuwd/+Tiuo0wfB/1ADgqoDqnit5XvaX7R37q4bu+yLxrxTf9njvytO7gUzm5Z1AGqWnmr5Q5q
hX2+VUQn+DhmMx01YShVb7ICJxfuZeHkEujufHILXmJ7wZUfhwEuBTSb58cnr96eQkH5lE1LwbOmzesdp4Iyzi2VpBIRST7WnR2lAvbR1x1m9wHDZLcf9iXc
PmBMquaz9YWdYn7GPoycsaiyNNLxfTN0/2QQvyX2emku/QRb8fJtg4kLjxuQGnebC924/oMqznfCwchIXVCBa0nMxKXtJ1eMbWixkU2q7oO62VPC12AZYrTF
58GZr51aTPSNKQSlHPJt5Fh49MJLr2KIFK9wvTj1y8vvDMnCpfzzRklC3VaBBKziYAeO+N7JldQqnKpuJqz5guGuGKiN5Yu2VaEhFU6oZY8xvvA6O3ldR944
dZaTXcqd399dcSNrqYg0TAfltl51dOwVQVyYGl7QIHC1pvLHmoaLiRdbuOoSS7GduFOvb7gNJQT5yRxGosKF2DQbfMTjHR4ucIjkBhyXprZNEHpR2HHf4wQm
qJ5TbFtgATk289QSANDfYvByZybjFGpjJ+5yqBn4Or7I3dOyaeerazycmQaZketSv7xTu7kz1+k89M7OGvvvCnPIfGs1twk/cdwr11/Atza7ek/jIuro1R1f
erBppCJBYI+8YDxfujFip7NOVlYhHPA+8eZCu42db3aglUqJxbXpFhiJKVClXt7eZXacE9wEn4eLKXA7v2sOOf66r01E9c65sxtapcujtZ+a0+ZPbUQYoYUK
PG13O1z8lv65K9/XdlvUl908RxxDlLoAAIwkGBCOg/jzfhg4TitzSFGf1AeGqk4X2DlB2Wfr1L05IyD39anAJT7KWR/x78yOtdULRtNGl+pk6KJR/EUPGYI0
LDhHT3bZxdljA03MVpNnAC+fkFdRBnrPHsuVZ2SP/8JKU9YHk0Z+Eh03cBe3wxEEs3sH629ipAyfvYnxIwzSPnKwFy0WTxUxMFUidZpUoCrc7z0r2QFbpYsL
s94q3p9qtyVmF9hpKMZmaVXBEl1MJlbp5tEtVdYDAjeIfgYOhQdRNQpX37h9cnJwdvYEI4rTwRNQmCdDwZAnrw/evstvU/w4qxX80nhj0ynhWIo4qOK1q2hT
8ynihFFvORAeiSZNRtTyNw1+eRhFAAX52UF80SxqfiiFY3IrsRQR1Urf17MeZktsBYWwIvZMCkEsLz7hu1ekh1PpIrtp5TJf5gw1vqzBChlGyVueMubasUo+
zCFX3leYxG7Myh9M87a+03yeZiCHWg/B+lQPi6sXmVy78YxF9uTToHl08uHwzZnm2Cs+wQDnRkzSsqOwVLOpFt/ivj5rqRItJisl73XBHHQ6h2jJmZ0hBfmE
/t9ml55ZCWVCpfwtAxaoPRPwaw7bueMK86ZB872jO3Zo4y95L28OfDZg03M+Tnt4K+qkcNPxU1WpzOH3tWfISnljWWqjUk4mQM5ma/y31ZCKSnnVMGL8uG8Q
JxbEcI4PPubRscEvZJ324PqsYU/drSn59b+1aM2CGQjP0Sugdgmp1U5ak8xwaBaq+IIzH+dvEPHctb9cLStufzl2f+EQyutoCxGePYgGm9kSlKWgrcFtxL1i
56vm7mGhOx4SrqPo9gar0OdDYxKzWh6xbaq0ORCuMKBagXr6GlB2jq0C6TYINuIPdpRxUEcPy1eptRzbZgCdqkbDa4LcaARdY2N5j7RdJt9fXjhw5Kv15OjU
OXxH4OElK3bQmY9XYZ+vmf5P5DC1MMWCbQeuiOBeKpcriAWRsE3d0r5oNQ9LoN4PD95tdCCul1agMdwQU3TLjCLA6NIYeTilSrJlBuEOHjnXCFLI+x5vsiG7
6A1sWS+50jIZILTkkANZMylMIuuziyla8PnlIAtLyfOCQMF5AoFbf49E2NRTIEToJfjQpllRDuf6D72JPClvJEvRm8nTpKEt1UoJh0n2RjEtWRnFlcahkmrr
V8ulBJhOTQ3b07jaob4QESHV5Zu3Af9iDQaiQVEkw7lFrrtmeYm2jPAyO66U5Vq9YyHkS1aggs+ukMY2tsw/VIhZPS5IOSSIXil3FmxZgFCuScxu6g1HFRYW
dr2LUm2Y9Xsr083qpC7Y1W2rTvCuYgrBVwZorhdq3atDpPfZKZ86MamlOttGWjqb5O4lpcp/5sonVe+M+kDxfB5G6JPBG+q1OE1wdm3maCWcAgh1T8eqpltM
ADPbLa4aOgSpWK2LomNndjL3I5NqbBkF7lQYGrJAx8M3yd8Z5m01JkPf7iAWgtX8zH5hqgnTcsCdKF4eRDclkV9DdxiNZiZh0y37AJx9IrbCtLV5cK5q6eQU
/+zg+OTd0Vk+W/zKwaQUDysd5NUYe1mlDferD++PyuSawZOk8ZRB1Pzm39vfLNvfTM6/edP/5rj/zdn/JDLBeWZLAWL57A/WgH95/D3+Hn+Pv8ff4+/x9/h7
/D3+Hn+Pv8ff4+/x9/h7/D3+Hn+Pv8ff4+/x9/h7/D3+Hn+Pv8ff4+/x9/h7/D3+/ov8/i+B23cGAMgAAA==
"""
EXPECTED_SHA = {
 "w3_spec.py": "c6decb8e07b64261b7636e4ec2e87c95e11571e7b1fc7d8f31f64648c77339da",
 "w3_model.py": "a274bbff7db6501c971841892ef674c72bfe0435058254fd652ce5f838977c64",
 "w3_core.py": "79607f7d7749610523a7b6e85500a2d9557e2ad9b44afde46a467b4aabe3888b"
}
PKG = Path('/content/w3pkg')
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz') as tf:
    tf.extractall(PKG)
CODE_SHA = {n: hashlib.sha256((PKG / 'windows' / n).read_bytes()).hexdigest() for n in EXPECTED_SHA}
assert CODE_SHA == EXPECTED_SHA, f'G-hash: extracted code differs: {CODE_SHA}'
if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
for m in [m for m in sys.modules if m == 'windows' or m.startswith('windows.')]:
    del sys.modules[m]
from windows import w3_spec as S, w3_core as C
print('G-hash OK:', {k: v[:12] for k, v in CODE_SHA.items()})


In [ ]:
#@title Data: CIFAR-10 (binary), md5-checked; cached on Drive after the first download
import shutil, urllib.request
LOCAL = Path('/content/cifar-10-binary.tar.gz')
CACHE = W3 / 'cifar-10-binary.tar.gz'
if not (LOCAL.exists() and C.md5_file(LOCAL) == S.CIFAR_MD5):
    if CACHE.exists() and C.md5_file(CACHE) == S.CIFAR_MD5:
        shutil.copyfile(CACHE, LOCAL)
    else:
        req = urllib.request.Request(S.CIFAR_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=600) as r, open(LOCAL, 'wb') as f:
            shutil.copyfileobj(r, f)
        assert C.md5_file(LOCAL) == S.CIFAR_MD5, 'G-data: CIFAR-10 md5 mismatch'
        shutil.copyfile(LOCAL, CACHE)
print('CIFAR-10 OK', C.md5_file(LOCAL))


In [ ]:
#@title Run (training, gates, harvest; resume-safe)
stage = 'smoke' if SMOKE else 'flight'
stamp = RESUME_STAMP.strip() or time.strftime('%Y%m%d_%H%M')
OUT = W3 / f'{stage}_{stamp}'
OUT.mkdir(parents=True, exist_ok=True)
print(f'STAMP {stamp}  ->  MyDrive/semcore/w3/{OUT.name}   (paste the stamp into RESUME_STAMP to resume)')
LOG = OUT / 'run_log.txt'
def log(msg):
    line = time.strftime('%H:%M:%S ') + str(msg)
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')
log(f'start {stage}; code {json.dumps({k: v[:12] for k, v in CODE_SHA.items()})}')
C.run(OUT, LOCAL, smoke=SMOKE, device=torch.device('cuda'), log=log, code_sha=CODE_SHA)
log('run finished')


In [ ]:
#@title Summary
man = json.loads((OUT / 'manifest.json').read_text()) if (OUT / 'manifest.json').exists() else []
sets = sorted({x['set'] for x in man})
print(f'{OUT.name}: {len(sets)} sets in {len(man)} parts, largest part {max([x["bytes"] for x in man] or [0]) / 1e6:.2f} MB')
for tag in ('on', 'drop', 'off'):
    g = OUT / f'gate_{tag}.json'
    if g.exists():
        gg = json.loads(g.read_text())
        print(f'  gate {tag}: {"PASS" if gg["pass"] else "FAIL"} at {gg["epochs"]} epochs; ' +
              ' '.join(f'{b["steps"][0]}-{b["steps"][1]}: {b["mse_model"]:.4f}<{b["mse_best_multiple"]:.4f}' for b in gg['bands']))
for f in ('gdet.json', 'resume_test.json'):
    if (OUT / f).exists():
        print(' ', f, json.loads((OUT / f).read_text()).get('pass'))
print('DONE' if (OUT / 'DONE').exists() else 'NOT DONE')
if SMOKE:
    print('\nSmoke finished. Leave this runtime connected. After Claude checks it, set SMOKE = False and Run all again.')
